# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tokihab/FlyRank-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row means: A single content page (URL) snapshot for a specific month.

Table used: The main internship warehouse dataset from Hugging Face.

Time window: The mid-panel month of March 2026 (2026-03).

Proxy: Unsupervised clustering to group URLs into archetypes based on traffic, engagement, and word count.

Deliberately excluded: Any metrics from April 2026 or later (future clicks, future success flags) to prevent data leakage.

In [4]:
import pandas as pd
import pyarrow.compute as pc
import datetime
from datasets import load_dataset
from google.colab import userdata

print("⚡ 1. Loading and Filtering natively via PyArrow...")
hf_token = userdata.get('HF_TOKEN')

ds_fact = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", split="train", token=hf_token)

# Filter natively before pandas
arrow_table = ds_fact.data.table
date_filter = (
    (pc.field("report_date") >= datetime.date(2026, 3, 1)) &
    (pc.field("report_date") < datetime.date(2026, 4, 1))
)
df_mar_daily = arrow_table.filter(date_filter).to_pandas()

print("⚡ 2. Loading dimensions and Merging (Vectorized)...")
df_dim = load_dataset("FlyRank/internship-warehouse", "dim_content", split="train", token=hf_token).to_pandas()

# Optimized Aggregation
df_mar_fact = df_mar_daily.groupby('content_hash_id', observed=True).agg({
    'gsc_clicks': 'sum',
    'gsc_impressions': 'sum',
    'gsc_avg_position': 'mean',
}, as_index=False)
df_mar_fact.rename(columns={'gsc_clicks': 'clicks', 'gsc_impressions': 'impressions', 'gsc_avg_position': 'position'}, inplace=True)

# FIXED MERGE: We only pull 'content_hash_id' and 'word_count'. The 'url' column doesn't exist.
df_mar = df_mar_fact.merge(df_dim[['content_hash_id', 'word_count']], on='content_hash_id', how='left')

# Since it's anonymized, we'll set our 'url' column to the hash ID so the rest of your notebook works
df_mar['url'] = df_mar['content_hash_id']

# Vectorized Fallbacks and Metrics
df_mar['ctr'] = (df_mar['clicks'] / df_mar['impressions']).fillna(0.0)
df_mar['month'] = pd.Timestamp('2026-03-01')

print(f"\n✓ Grain Check (one row = one URL/month):")
print(f"  - Total rows: {len(df_mar):,}")
print(f"  - Is grain strict (unique URLs)? {df_mar['url'].is_unique}")

⚡ 1. Loading and Filtering natively via PyArrow...


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

⚡ 2. Loading dimensions and Merging (Vectorized)...

✓ Grain Check (one row = one URL/month):
  - Total rows: 331,437
  - Is grain strict (unique URLs)? True


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

The 5-Feature Frame:

    clicks: Knowable at the decision moment because it represents historical 30-day search traffic.

    impressions: Knowable at the decision moment because it represents historical search visibility.

    ctr: Knowable at the decision moment because it is derived from past clicks and impressions.

    position: Knowable at the decision moment because it reflects the average historical ranking.

    word_count: Knowable at the decision moment because it is a static property of the published page.

Context fields: url, month.
Excluded: future_clicks, future_success_flag (Anything that hasn't happened yet at the end of March).

In [5]:
features = ['clicks', 'impressions', 'ctr', 'position', 'word_count']
context = ['url', 'month']

# Availability check
df_mar['is_active'] = df_mar['impressions'] > 0
active_count = df_mar['is_active'].sum()
survival_pct = (active_count / len(df_mar)) * 100
print(f"✓ Availability (is_active = impressions > 0):")
print(f"  - Active rows: {active_count:,} / {len(df_mar):,} ({survival_pct:.1f}%)\n")

# Slice the clean feature frame once
df_features = df_mar[df_mar['is_active']][context + features].copy()
print(f"✓ Final feature frame created. Shape: {df_features.shape}")
display(df_features.head())

✓ Availability (is_active = impressions > 0):
  - Active rows: 176,738 / 331,437 (53.3%)

✓ Final feature frame created. Shape: (176738, 7)


,url,month,clicks,impressions,ctr,position,word_count
0,content_000005d4ced12088,2026-03-01,0,86,0.000000,72.854861,NaN
2,content_00007bd2985b77c3,2026-03-01,0,47,0.000000,5.269565,NaN
6,content_0000cd28fbda69f3,2026-03-01,0,29,0.000000,4.251282,990.0
8,content_0000d495bfbfb4a8,2026-03-01,0,15,0.000000,3.333333,2832.0
11,content_00014efc121d911d,2026-03-01,1,116,0.008621,4.964683,NaN


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

To demonstrate leakage, I will intentionally inject a future metric (simulating next month's clicks) and a future success flag. In a clustering task, adding a future label perfectly separates the data into "successful" and "failing" clusters, giving a flawless but entirely fake Silhouette score. I will show the trap, then delete the illegal columns to keep the data honest.

In [9]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

print("⚠️ TRAP: Adding future-knowledge columns...")

# FIXED: Sample down to 10,000 rows!
# Silhouette score is O(N^2) and will hang forever on 176k rows.
df_sample = df_features.sample(n=10000, random_state=42).copy()

df_sample['future_clicks'] = df_sample['clicks'] * 1.5
df_sample['future_success_flag'] = (df_sample['future_clicks'] > 500).astype(int)

# Scale & Cluster WITH illegal columns
X_illegal = StandardScaler().fit_transform(df_sample[features + ['future_clicks', 'future_success_flag']].fillna(0))

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
illegal_score = silhouette_score(X_illegal, kmeans.fit_predict(X_illegal))
print(f"  Silhouette Score WITH LEAKAGE: {illegal_score:.4f} (Artificially high!)")

print("\n✅ FIX: Removing future-knowledge columns...")
# Re-cluster with honest data
X_honest = StandardScaler().fit_transform(df_sample[features].fillna(0))
honest_score = silhouette_score(X_honest, kmeans.fit_predict(X_honest))
print(f"  Silhouette Score on HONEST data: {honest_score:.4f} (Real performance)")
print(f"  📊 Score gap (leakage impact): {illegal_score - honest_score:.4f}")

⚠️ TRAP: Adding future-knowledge columns...
  Silhouette Score WITH LEAKAGE: 0.3515 (Artificially high!)

✅ FIX: Removing future-knowledge columns...
  Silhouette Score on HONEST data: 0.3662 (Real performance)
  📊 Score gap (leakage impact): -0.0147


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Unbalanced history. Early rows in the dataset may only contain Google Search Console (GSC) data without complete on-page engagement metrics (like bounce rate or scroll depth). This means our clustering for older months might skew heavily toward search visibility rather than actual user engagement, limiting our ability to define archetypes based on how users actually read the content.

In [8]:
# Missing values check (Proving Data Limits / GSC-only rows)
print(f"✓ Missing values check (identifying data limits):")
missing = df_features[features].isnull().sum()
if missing.sum() == 0:
    print("  - No missing values ✓")
else:
    print(missing[missing > 0])

✓ Missing values check (identifying data limits):
word_count    55315
dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.